# Highest Cost Orders
https://platform.stratascratch.com/coding/9915-highest-cost-orders?code_type=6

In [0]:
from pyspark.sql.types import StructType, StructField, LongType, StringType

customers_schema = StructType([
    StructField("id", LongType(), True),
    StructField("first_name", StringType(), True),
    StructField("last_name", StringType(), True),
    StructField("city", StringType(), True),
    StructField("address", StringType(), True),
    StructField("phone_number", StringType(), True)
])

customers_data = [
    (1, "Rohan", "Sharma", "Delhi", "12 Karol Bagh", "9876543210"),
    (2, "Amit", "Verma", "Mumbai", "45 Andheri West", "9823456781"),
    (3, "Sneha", "Iyer", "Bangalore", "22 MG Road", "9811122233"),
    (4, "Priya", "Singh", "Kolkata", "78 Park Street", "9898989898"),
    (5, "Rahul", "Das", "Hyderabad", "90 Jubilee Hills", "9776655443"),
    (6, "Neha", "Kapoor", "Delhi", "33 Saket", "9123456789"),
    (7, "Arjun", "Mehta", "Pune", "11 Koregaon Park", "9001122334"),
    (8, "Kavya", "Nair", "Chennai", "56 T Nagar", "9988776655")
]

customers = spark.createDataFrame(customers_data, customers_schema)


# Orders schema
from pyspark.sql.types import DateType
from datetime import date

orders_schema = StructType([
    StructField("id", LongType(), True),
    StructField("cust_id", LongType(), True),
    StructField("order_date", DateType(), True),
    StructField("order_details", StringType(), True),
    StructField("total_order_cost", LongType(), True)
])

orders_data = [

    # 🔹 Date: 2019-02-01 (tie case)
    (1, 1, date(2019, 2, 1), "Shoes", 50),
    (2, 1, date(2019, 2, 1), "Shirt", 50),   # total = 100

    (3, 2, date(2019, 2, 1), "Jacket", 100), # total = 100 (tie)

    (4, 3, date(2019, 2, 1), "Cap", 20),

    # 🔹 Date: 2019-02-02 (clear winner)
    (5, 1, date(2019, 2, 2), "Shoes", 40),
    (6, 2, date(2019, 2, 2), "Laptop", 500),  # winner
    (7, 3, date(2019, 2, 2), "Bag", 100),

    # 🔹 Date: 2019-02-03 (multiple orders same customer)
    (8, 3, date(2019, 2, 3), "Tablet", 200),
    (9, 3, date(2019, 2, 3), "Keyboard", 100), # total = 300 (winner)

    (10, 4, date(2019, 2, 3), "Mouse", 150),

    # 🔹 Date: 2019-03-01 (tie again)
    (11, 4, date(2019, 3, 1), "TV", 400),
    (12, 5, date(2019, 3, 1), "Phone", 400),

    # 🔹 Date: 2019-03-02 (aggregation trick)
    (13, 2, date(2019, 3, 2), "Shoes", 100),
    (14, 2, date(2019, 3, 2), "Watch", 200),  # total = 300 (winner)

    (15, 1, date(2019, 3, 2), "Bag", 250),

    # 🔹 Date: 2019-04-01 (single dominant)
    (16, 5, date(2019, 4, 1), "Laptop", 1000), # winner

    (17, 3, date(2019, 4, 1), "Shoes", 200),

    # 🔹 Outside filter (should be ignored)
    (18, 1, date(2019, 5, 10), "Phone", 9999),
]

orders = spark.createDataFrame(orders_data, orders_schema)

In [0]:
customers.show(5)

In [0]:
orders.show()

In [0]:
# Import your libraries
import pyspark
import pyspark.sql.functions as F
# Start writing code
df_joined = customers.join(orders, customers["id"] == orders["cust_id"],how="inner")
df_joined = df_joined.filter((F.col("order_date") >= "2019-02-01") & (F.col("order_date") <= "2019-05-01"))
df_1 = df_joined.select("first_name","order_date","total_order_cost")
df_1.show(truncate=False)

In [0]:
df_1.printSchema()

In [0]:
df_2 = df_1.groupBy("first_name","order_date").agg(F.sum("total_order_cost").alias("total_order_cost"))
df_2.show(truncate=False)

In [0]:
df_3 = df_2.groupBy("order_date").agg(F.max("total_order_cost"))
df_3.show()

In [0]:
from pyspark.sql.window import Window
w = Window.partitionBy("order_date").orderBy(F.desc("total_order_cost"))

df_4 = df_2.withColumn("rank",F.dense_rank().over(w)).filter("rank == 1").drop("rank")
df_4.show()

In [0]:
# df_joined = df_joined.groupBy("first_name","order_date").agg({"total_order_cost":"sum"})
df_joined_sum = df_joined.groupBy("first_name","order_date").agg(F.sum("total_order_cost").alias("total_order_cost"))\
    .orderBy(F.asc("order_date"))
df_joined_sum.show()